In [ ]:
import pandas as pd
import plotly.express as px
from datetime import datetime, time
from pytz import timezone

print("📊 App Size vs Rating Analysis")
print("Bubble size represents number of installs")

# ==================================================
# TIME RESTRICTION (5 PM – 7 PM IST)
# ==================================================
ist = timezone("Asia/Kolkata")
current_time = datetime.now(ist).time()

if not (time(17, 0) <= current_time <= time(19, 0)):
    print("⏰ Dashboard is accessible only between 5 PM – 7 PM IST")
else:
    print("✅ Dashboard accessible between 5 PM – 7 PM IST")

    # ==================================================
    # LOAD DATA (Play Store + User Reviews)
    # ==================================================
    def load_data():
        play_df = pd.read_csv("Play Store Data.csv")
        review_df = pd.read_csv("User Reviews.csv")

        # -----------------------------
        # CLEAN PLAY STORE DATA
        # -----------------------------
        play_df["Rating"] = pd.to_numeric(play_df["Rating"], errors="coerce")
        play_df["Reviews"] = pd.to_numeric(play_df["Reviews"], errors="coerce")

        play_df["Installs"] = (
            play_df["Installs"]
            .astype(str)
            .str.replace("[+,]", "", regex=True)
        )
        play_df["Installs"] = pd.to_numeric(play_df["Installs"], errors="coerce")

        play_df["Size_MB"] = (
            play_df["Size"]
            .astype(str)
            .str.replace("M", "", regex=False)
            .replace(["Varies with device"], None)
        )
        play_df["Size_MB"] = pd.to_numeric(play_df["Size_MB"], errors="coerce")

        # -----------------------------
        # CLEAN USER REVIEWS DATA
        # -----------------------------
        review_df["Sentiment_Subjectivity"] = pd.to_numeric(
            review_df["Sentiment_Subjectivity"], errors="coerce"
        )

        # -----------------------------
        # MERGE DATASETS
        # -----------------------------
        merged_df = play_df.merge(
            review_df[["App", "Sentiment_Subjectivity"]],
            on="App",
            how="left"
        )

        return merged_df

    df = load_data()

    # ==================================================
    # FILTER CONDITIONS
    # ==================================================
    allowed_categories = [
        "GAME", "BEAUTY", "BUSINESS", "COMICS",
        "COMMUNICATION", "DATING", "ENTERTAINMENT",
        "SOCIAL", "EVENTS"
    ]

    df["Category"] = df["Category"].astype(str).str.upper()

    filtered_df = df[
        (df["Rating"] > 3.5) &
        (df["Reviews"] > 500) &
        (df["Installs"] > 50000) &
        (df["Sentiment_Subjectivity"] > 0.5) &
        (~df["App"].astype(str).str.contains("S", case=False, na=False)) &
        (df["Category"].isin(allowed_categories))
    ].dropna(subset=["Size_MB", "Rating", "Installs"])

    # ==================================================
    # CATEGORY TRANSLATION (GRAPH ONLY)
    # ==================================================
    translation_map = {
        "BEAUTY": "सौंदर्य",      
        "BUSINESS": "வணிகம்",     
        "DATING": "Partnersuche"
    }

    filtered_df["Category_Display"] = filtered_df["Category"].replace(translation_map)

    # ==================================================
    # GAME CATEGORY HIGHLIGHT
    # ==================================================
    filtered_df["Highlight"] = filtered_df["Category"].apply(
        lambda x: "Game" if x == "GAME" else "Other"
    )

    # ==================================================
    # BUBBLE CHART
    # ==================================================
    fig = px.scatter(
        filtered_df,
        x="Size_MB",
        y="Rating",
        size="Installs",
        color="Highlight",
        color_discrete_map={
            "Game": "pink",
            "Other": "#7EC8E3"
        },
        hover_name="App",
        hover_data={
            "Installs": True,
            "Reviews": True,
            "Category_Display": True,
            "Sentiment_Subjectivity": True
        },
        labels={
            "Size_MB": "App Size (MB)",
            "Rating": "Average Rating"
        },
        title="App Size vs Average Rating"
    )

    fig.update_layout(
        showlegend=False,
        plot_bgcolor="white",
        xaxis=dict(showgrid=True, gridcolor="lightgrey"),
        yaxis=dict(showgrid=True, gridcolor="lightgrey")
    )

    fig.show()

    # ==================================================
    # FOOTER (Console Display)
    # ==================================================
    print("\nFilters Applied:")
    print("✔ Rating > 3.5")
    print("✔ Reviews > 500")
    print("✔ Installs > 50K")
    print("✔ Sentiment Subjectivity > 0.5")
    print('✔ App name does not contain letter "S"')
    print("✔ Selected categories only")
    print("✔ Game category highlighted in pink")
